# Transaction aggregation and additive player-value export

This notebook aggregates transaction-side features and exports the player-level inputs used by the additive production-value model.


In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load player features, transaction scope, and feature-audit inputs.
transaction_Player_Features = pd.read_parquet("../data/interim/transaction_player_features_offseason.parquet")

transactions_In_Scope = pd.read_parquet("../data/interim/transactions_in_scope_offseason.parquet")

transaction_Feature_Audit = pd.read_parquet("../data/interim/transaction_Feature_Audit_offseason.parquet")

In [3]:
# Create a stable transaction-row identifier for all joins.
transactions_In_Scope = transactions_In_Scope.reset_index(drop=True)
transactions_In_Scope.insert(0, "transaction_row_id", transactions_In_Scope.index.to_numpy())

In [4]:
# Define counting and rate metrics used in side aggregation.
counting_stat_names = [
    "points",
    "assists",
    "turnovers",
    "offensive_rebounds",
    "defensive_rebounds",
    "total_rebounds",
    "steals",
    "blocks",
    "personal_fouls",
    "field_goals_made",
    "field_goals_attempted",
    "three_pointers_made",
    "three_pointers_attempted",
    "free_throws_made",
    "free_throws_attempted",
    "plus_minus",
]

advanced_rate_metrics = [
    "court_time_percentage",
    "usage_percentage",
    "assist_percentage",
    "turnover_percentage",
    "offensive_rebound_percentage",
    "defensive_rebound_percentage",
    "total_rebound_percentage",
    "steal_percentage",
    "block_percentage",
]

top_player_metrics = [
    "minutes_per_game",
    "points_per_100",
    "assists_per_100",
    "total_rebounds_per_100",
    "true_shooting_percentage",
    "effective_field_goal_percentage",
    "usage_percentage",
    "game_score_average",
]

In [5]:
# Aggregate multiple players into one transaction-side feature vector.
def aggregate_transaction_side_features(side_rows, side_prefix):
    """
    Aggregate all player-level feature rows on one side of a
    team-centric transaction.

    Parameters
    ----------
    side_rows : pandas.DataFrame
        All acquired or relinquished player rows associated
        with one transaction_row_id.

    side_prefix : str
        Prefix identifying the transaction side, normally
        'acquired_' or 'relinquished_'.

    Returns
    -------
    dict
        Flat dictionary of aggregated transaction-side features.
    """

    if not side_prefix.endswith("_"):
        side_prefix = f"{side_prefix}_"

    output = {
        f"{side_prefix}listed_player_count": len(side_rows),
        f"{side_prefix}matched_player_count": int(side_rows["player_id"].notna().sum()),
        f"{side_prefix}calculated_player_count": int(side_rows["feature_status"].eq("calculated").sum()),
        f"{side_prefix}failed_player_count": int(side_rows["feature_status"].ne("calculated").sum()),
        f"{side_prefix}player_id_missing_count": int(side_rows["feature_status"].eq("player_id_missing").sum()),
        f"{side_prefix}no_prior_appearance_count": int(side_rows["feature_status"].eq("no_prior_box_score_appearance").sum()),
        f"{side_prefix}exact_match_count": int(side_rows["player_match_status"].eq("exact_match").sum()),
        f"{side_prefix}unresolved_name_count": int(side_rows["player_match_status"].eq("unresolved_name").sum()),
        f"{side_prefix}no_prior_nba_appearance_count": int(side_rows["player_match_status"].eq("no_prior_nba_appearance").sum()),
        f"{side_prefix}player_names": " | ".join(side_rows["transaction_player_name"].dropna().astype(str)),
    }

    calculated_rows = side_rows.loc[side_rows["feature_status"].eq("calculated")].copy()

    output[f"{side_prefix}has_calculated_player"] = not calculated_rows.empty

    if calculated_rows.empty:
        return output

    output.update(
        {
            f"{side_prefix}current_season_appearance_count": int(calculated_rows["has_current_season_appearance"].eq(True).sum()),
            f"{side_prefix}career_seasons_played_mean": (calculated_rows["career_seasons_played"].mean()),
            f"{side_prefix}career_seasons_played_max": (calculated_rows["career_seasons_played"].max()),
        }
    )

    # ---------------------------------------------------------
    # Aggregate season, career, and last-10 scopes
    # ---------------------------------------------------------

    for scope in ["season", "career", "last10"]:

        games_column = f"{scope}_games_played"
        starts_column = f"{scope}_games_started"
        minutes_column = f"{scope}_minutes"
        possessions_column = f"{scope}_estimated_player_possessions"

        if games_column not in calculated_rows.columns:
            continue

        scope_rows = calculated_rows.loc[calculated_rows[games_column].notna()].copy()

        output[f"{side_prefix}{scope}_players_with_data"] = len(scope_rows)

        if scope_rows.empty:
            continue

        games_played = scope_rows[games_column].sum(min_count=1)

        games_started = scope_rows[starts_column].sum(min_count=1)

        minutes = scope_rows[minutes_column].sum(min_count=1)

        estimated_possessions = scope_rows[possessions_column].sum(min_count=1)

        output.update(
            {
                f"{side_prefix}{scope}_games_played_sum": games_played,
                f"{side_prefix}{scope}_games_started_sum": games_started,
                f"{side_prefix}{scope}_minutes_sum": minutes,
                f"{side_prefix}{scope}_estimated_player_possessions_sum": (estimated_possessions),
                f"{side_prefix}{scope}_minutes_per_game": (
                    minutes / games_played if pd.notna(games_played) and games_played > 0 else np.nan
                ),
                f"{side_prefix}{scope}_start_percentage": (
                    100 * games_started / games_played if pd.notna(games_played) and games_played > 0 else np.nan
                ),
            }
        )

        # -----------------------------------------------------
        # Games-weighted win percentage
        # -----------------------------------------------------

        win_percentage_column = f"{scope}_win_percentage_when_appearing"

        valid_win_rows = scope_rows[win_percentage_column].notna() & scope_rows[games_column].gt(0)

        output[f"{side_prefix}{scope}_win_percentage_when_appearing"] = (
            np.average(scope_rows.loc[valid_win_rows, win_percentage_column], weights=scope_rows.loc[valid_win_rows, games_column])
            if valid_win_rows.any()
            else np.nan
        )

        # -----------------------------------------------------
        # Counting statistics
        # -----------------------------------------------------

        scope_totals = {}

        for statistic in counting_stat_names:
            statistic_column = f"{scope}_{statistic}"

            if statistic_column not in scope_rows.columns:
                continue

            statistic_total = scope_rows[statistic_column].sum(min_count=1)

            scope_totals[statistic] = statistic_total

            output[f"{side_prefix}{scope}_{statistic}_sum"] = statistic_total

            output[f"{side_prefix}{scope}_{statistic}_per_game"] = (
                statistic_total / games_played if pd.notna(statistic_total) and pd.notna(games_played) and games_played > 0 else np.nan
            )

            output[f"{side_prefix}{scope}_{statistic}_per_36"] = (
                36 * statistic_total / minutes if pd.notna(statistic_total) and pd.notna(minutes) and minutes > 0 else np.nan
            )

            output[f"{side_prefix}{scope}_{statistic}_per_100"] = (
                100 * statistic_total / estimated_possessions
                if pd.notna(statistic_total) and pd.notna(estimated_possessions) and estimated_possessions > 0
                else np.nan
            )

        # -----------------------------------------------------
        # Recalculate shooting rates from pooled totals
        # -----------------------------------------------------

        field_goals_made = scope_totals.get("field_goals_made", np.nan)

        field_goal_attempts = scope_totals.get("field_goals_attempted", np.nan)

        three_pointers_made = scope_totals.get("three_pointers_made", np.nan)

        three_point_attempts = scope_totals.get("three_pointers_attempted", np.nan)

        free_throws_made = scope_totals.get("free_throws_made", np.nan)

        free_throw_attempts = scope_totals.get("free_throws_attempted", np.nan)

        points = scope_totals.get("points", np.nan)
        assists = scope_totals.get("assists", np.nan)
        turnovers = scope_totals.get("turnovers", np.nan)

        two_pointers_made = (
            field_goals_made - three_pointers_made if pd.notna(field_goals_made) and pd.notna(three_pointers_made) else np.nan
        )

        two_point_attempts = (
            field_goal_attempts - three_point_attempts if pd.notna(field_goal_attempts) and pd.notna(three_point_attempts) else np.nan
        )

        true_shooting_denominator = (
            2 * (field_goal_attempts + 0.44 * free_throw_attempts)
            if pd.notna(field_goal_attempts) and pd.notna(free_throw_attempts)
            else np.nan
        )

        output.update(
            {
                f"{side_prefix}{scope}_field_goal_percentage": (
                    field_goals_made / field_goal_attempts if pd.notna(field_goal_attempts) and field_goal_attempts > 0 else np.nan
                ),
                f"{side_prefix}{scope}_two_point_percentage": (
                    two_pointers_made / two_point_attempts if pd.notna(two_point_attempts) and two_point_attempts > 0 else np.nan
                ),
                f"{side_prefix}{scope}_three_point_percentage": (
                    three_pointers_made / three_point_attempts if pd.notna(three_point_attempts) and three_point_attempts > 0 else np.nan
                ),
                f"{side_prefix}{scope}_free_throw_percentage": (
                    free_throws_made / free_throw_attempts if pd.notna(free_throw_attempts) and free_throw_attempts > 0 else np.nan
                ),
                f"{side_prefix}{scope}_effective_field_goal_percentage": (
                    (field_goals_made + 0.5 * three_pointers_made) / field_goal_attempts
                    if pd.notna(field_goal_attempts) and field_goal_attempts > 0
                    else np.nan
                ),
                f"{side_prefix}{scope}_true_shooting_percentage": (
                    points / true_shooting_denominator if pd.notna(true_shooting_denominator) and true_shooting_denominator > 0 else np.nan
                ),
                f"{side_prefix}{scope}_three_point_attempt_rate": (
                    three_point_attempts / field_goal_attempts if pd.notna(field_goal_attempts) and field_goal_attempts > 0 else np.nan
                ),
                f"{side_prefix}{scope}_free_throw_rate": (
                    free_throw_attempts / field_goal_attempts if pd.notna(field_goal_attempts) and field_goal_attempts > 0 else np.nan
                ),
                f"{side_prefix}{scope}_assist_turnover_ratio": (assists / turnovers if pd.notna(turnovers) and turnovers > 0 else np.nan),
            }
        )

        # -----------------------------------------------------
        # Recalculate pooled Game Score values
        # -----------------------------------------------------

        game_score_column = f"{scope}_game_score_total"

        if game_score_column in scope_rows.columns:
            game_score_total = scope_rows[game_score_column].sum(min_count=1)

            output.update(
                {
                    f"{side_prefix}{scope}_game_score_total_sum": (game_score_total),
                    f"{side_prefix}{scope}_game_score_average": (
                        game_score_total / games_played
                        if pd.notna(game_score_total) and pd.notna(games_played) and games_played > 0
                        else np.nan
                    ),
                    f"{side_prefix}{scope}_game_score_per_36": (
                        36 * game_score_total / minutes if pd.notna(game_score_total) and pd.notna(minutes) and minutes > 0 else np.nan
                    ),
                }
            )

        # -----------------------------------------------------
        # Minutes-weighted advanced rates
        # -----------------------------------------------------

        weights = pd.to_numeric(scope_rows[minutes_column], errors="coerce")

        for metric in advanced_rate_metrics:
            metric_column = f"{scope}_{metric}"

            if metric_column not in scope_rows.columns:
                continue

            metric_values = pd.to_numeric(scope_rows[metric_column], errors="coerce")

            valid_weighted_rows = metric_values.notna() & weights.notna() & weights.gt(0)

            output[f"{side_prefix}{scope}_{metric}_weighted_mean"] = (
                np.average(metric_values.loc[valid_weighted_rows], weights=weights.loc[valid_weighted_rows])
                if valid_weighted_rows.any()
                else np.nan
            )

        # -----------------------------------------------------
        # Strongest individual player on the side
        # -----------------------------------------------------

        for metric in top_player_metrics:
            metric_column = f"{scope}_{metric}"

            if metric_column not in scope_rows.columns:
                continue

            output[f"{side_prefix}{scope}_{metric}_max"] = pd.to_numeric(scope_rows[metric_column], errors="coerce").max()

        # -----------------------------------------------------
        # Recency
        # -----------------------------------------------------

        days_column = f"{scope}_days_since_last_game"

        if days_column in scope_rows.columns:
            days_values = pd.to_numeric(scope_rows[days_column], errors="coerce")

            output.update(
                {
                    f"{side_prefix}{scope}_days_since_last_game_min": (days_values.min()),
                    f"{side_prefix}{scope}_days_since_last_game_mean": (days_values.mean()),
                    f"{side_prefix}{scope}_days_since_last_game_max": (days_values.max()),
                }
            )

    return output

In [6]:
multi_player_side = (
    transaction_Player_Features.groupby(["transaction_row_id", "transaction_side"]).size().loc[lambda counts: counts > 1].index[0]
)

test_transaction_row_id = multi_player_side[0]
test_transaction_side = multi_player_side[1]

In [7]:
test_side_rows = transaction_Player_Features.loc[
    (transaction_Player_Features["transaction_row_id"] == test_transaction_row_id)
    & (transaction_Player_Features["transaction_side"] == test_transaction_side)
].copy()

In [8]:
test_side_prefix = "acquired_" if test_transaction_side == "Acquired" else "relinquished_"

In [9]:
test_side_features = aggregate_transaction_side_features(side_rows=test_side_rows, side_prefix=test_side_prefix)

pd.DataFrame(test_side_features.items(), columns=["feature", "value"])

,feature,value
0,acquired_listed_player_count,2
1,acquired_matched_player_count,2
2,acquired_calculated_player_count,2
3,acquired_failed_player_count,0
4,acquired_player_id_missing_count,0
...,...,...
321,acquired_last10_usage_percentage_max,14.926228
322,acquired_last10_game_score_average_max,3.5
323,acquired_last10_days_since_last_game_min,2.0
324,acquired_last10_days_since_last_game_mean,13.5


In [10]:
prefix = test_side_prefix

assert np.isclose(
    test_side_features[f"{prefix}season_points_per_100"],
    (100 * test_side_features[f"{prefix}season_points_sum"] / test_side_features[f"{prefix}season_estimated_player_possessions_sum"]),
    equal_nan=True,
)

In [11]:
assert np.isclose(
    test_side_features[f"{prefix}season_true_shooting_percentage"],
    (
        test_side_features[f"{prefix}season_points_sum"]
        / (
            2
            * (
                test_side_features[f"{prefix}season_field_goals_attempted_sum"]
                + 0.44 * test_side_features[f"{prefix}season_free_throws_attempted_sum"]
            )
        )
    ),
    equal_nan=True,
)

In [12]:
assert test_side_features[f"{prefix}listed_player_count"] == len(test_side_rows)

assert test_side_features[f"{prefix}calculated_player_count"] == test_side_rows["feature_status"].eq("calculated").sum()

print("Transaction-side aggregation validated.")

Transaction-side aggregation validated.


In [13]:
# Build one aggregate record for every transaction side.
transaction_side_feature_records = []

grouped_transaction_sides = transaction_Player_Features.groupby(["transaction_row_id", "transaction_side"], sort=False)

for (transaction_row_id, transaction_side), side_rows in grouped_transaction_sides:

    side_prefix = "acquired_" if transaction_side == "Acquired" else "relinquished_"

    side_features = aggregate_transaction_side_features(side_rows=side_rows, side_prefix=side_prefix)

    transaction_side_feature_records.append(
        {"transaction_row_id": transaction_row_id, "transaction_side": transaction_side, **side_features}
    )

transaction_Side_Features = pd.DataFrame(transaction_side_feature_records)

In [14]:
print("Player-level rows:", len(transaction_Player_Features))

print("Unique transaction sides:", transaction_Player_Features.groupby(["transaction_row_id", "transaction_side"]).ngroups)

print("Aggregated transaction-side rows:", len(transaction_Side_Features))

print("Aggregated feature columns:", len(transaction_Side_Features.columns))

Player-level rows: 6468
Unique transaction sides: 4424
Aggregated transaction-side rows: 4424
Aggregated feature columns: 654


In [15]:
transaction_Side_Features["transaction_side"].value_counts()

transaction_side
Acquired        2213
Relinquished    2211
Name: count, dtype: int64

In [16]:
duplicate_aggregated_sides = transaction_Side_Features.duplicated(subset=["transaction_row_id", "transaction_side"], keep=False)

print("Duplicate aggregated transaction sides:", duplicate_aggregated_sides.sum())

Duplicate aggregated transaction sides: 0


In [17]:
# Separate acquired-side and relinquished-side features.
acquired_columns = ["transaction_row_id"] + [column for column in transaction_Side_Features.columns if column.startswith("acquired_")]

relinquished_columns = ["transaction_row_id"] + [
    column for column in transaction_Side_Features.columns if column.startswith("relinquished_")
]

In [18]:
acquired_Transaction_Features = (
    transaction_Side_Features.loc[transaction_Side_Features["transaction_side"].eq("Acquired"), acquired_columns]
    .copy()
    .reset_index(drop=True)
)

relinquished_Transaction_Features = (
    transaction_Side_Features.loc[transaction_Side_Features["transaction_side"].eq("Relinquished"), relinquished_columns]
    .copy()
    .reset_index(drop=True)
)

In [19]:
print("Acquired-side rows:", len(acquired_Transaction_Features))

print("Acquired-side feature columns:", len(acquired_Transaction_Features.columns))

print("Relinquished-side rows:", len(relinquished_Transaction_Features))

print("Relinquished-side feature columns:", len(relinquished_Transaction_Features.columns))

Acquired-side rows: 2213
Acquired-side feature columns: 327
Relinquished-side rows: 2211
Relinquished-side feature columns: 327


In [20]:
aggregated_listed_player_count = (
    acquired_Transaction_Features["acquired_listed_player_count"].sum()
    + relinquished_Transaction_Features["relinquished_listed_player_count"].sum()
)

print("Player-level feature rows:", len(transaction_Player_Features))

print("Aggregated listed-player count:", aggregated_listed_player_count)

assert aggregated_listed_player_count == len(transaction_Player_Features)

Player-level feature rows: 6468
Aggregated listed-player count: 6468.0


In [21]:
aggregated_calculated_player_count = (
    acquired_Transaction_Features["acquired_calculated_player_count"].sum()
    + relinquished_Transaction_Features["relinquished_calculated_player_count"].sum()
)

source_calculated_player_count = transaction_Player_Features["feature_status"].eq("calculated").sum()

print("Source calculated players:", source_calculated_player_count)

print("Aggregated calculated players:", aggregated_calculated_player_count)

assert aggregated_calculated_player_count == source_calculated_player_count

Source calculated players: 5194
Aggregated calculated players: 5194.0


In [22]:
# Join both side aggregates back to the transaction grain.
transaction_Level_Features = transactions_In_Scope.merge(
    acquired_Transaction_Features, on="transaction_row_id", how="left", validate="one_to_one"
).merge(relinquished_Transaction_Features, on="transaction_row_id", how="left", validate="one_to_one")

In [23]:
print("Source transaction rows:", len(transactions_In_Scope))

print("Merged transaction rows:", len(transaction_Level_Features))

print("Unique transaction IDs:", transaction_Level_Features["transaction_row_id"].nunique())

assert len(transaction_Level_Features) == len(transactions_In_Scope)

assert transaction_Level_Features["transaction_row_id"].is_unique

Source transaction rows: 2749
Merged transaction rows: 2749
Unique transaction IDs: 2749


In [24]:
side_count_columns = [
    column
    for column in transaction_Level_Features.columns
    if (column.startswith("acquired_") or column.startswith("relinquished_"))
    and (column.endswith("_count") or column.endswith("_players_with_data"))
]

transaction_Level_Features[side_count_columns] = transaction_Level_Features[side_count_columns].fillna(0).astype("int64")

In [25]:
# Flag transactions with no listed players.
transaction_Level_Features["asset_only_transaction"] = transaction_Level_Features["acquired_listed_player_count"].eq(
    0
) & transaction_Level_Features["relinquished_listed_player_count"].eq(0)

In [26]:
# Require calculated players on both sides for production modeling.
transaction_Level_Features["player_feature_eligible"] = (
    transaction_Level_Features["acquired_has_calculated_player"] | transaction_Level_Features["relinquished_has_calculated_player"]
)

In [27]:
boolean_source_columns = ["acquired_has_calculated_player", "relinquished_has_calculated_player"]

for column in boolean_source_columns:
    transaction_Level_Features[column] = transaction_Level_Features[column].eq(True)

transaction_Level_Features["both_sides_have_calculated_players"] = (
    transaction_Level_Features["acquired_has_calculated_player"] & transaction_Level_Features["relinquished_has_calculated_player"]
)

In [28]:
transaction_Level_Features["only_acquired_side_has_calculated_players"] = (
    transaction_Level_Features["acquired_has_calculated_player"] & ~transaction_Level_Features["relinquished_has_calculated_player"]
)

In [29]:
transaction_Level_Features["only_relinquished_side_has_calculated_players"] = (
    ~transaction_Level_Features["acquired_has_calculated_player"] & transaction_Level_Features["relinquished_has_calculated_player"]
)

In [30]:
transaction_Level_Features["neither_side_has_calculated_players"] = (
    ~transaction_Level_Features["acquired_has_calculated_player"] & ~transaction_Level_Features["relinquished_has_calculated_player"]
)

In [31]:
coverage_category_count = (
    transaction_Level_Features["both_sides_have_calculated_players"].astype(int)
    + transaction_Level_Features["only_acquired_side_has_calculated_players"].astype(int)
    + transaction_Level_Features["only_relinquished_side_has_calculated_players"].astype(int)
    + transaction_Level_Features["neither_side_has_calculated_players"].astype(int)
)

coverage_category_count.value_counts()

1    2749
Name: count, dtype: int64

In [32]:
transaction_Level_Features[
    [
        "both_sides_have_calculated_players",
        "only_acquired_side_has_calculated_players",
        "only_relinquished_side_has_calculated_players",
        "neither_side_has_calculated_players",
    ]
].sum()

both_sides_have_calculated_players               1396
only_acquired_side_has_calculated_players         399
only_relinquished_side_has_calculated_players     399
neither_side_has_calculated_players               555
dtype: int64

In [33]:
print("Acquired sides with calculated players:", transaction_Level_Features["acquired_calculated_player_count"].gt(0).sum())

print("Relinquished sides with calculated players:", transaction_Level_Features["relinquished_calculated_player_count"].gt(0).sum())

print(
    "Transactions with either side calculated:",
    (
        transaction_Level_Features["acquired_calculated_player_count"].gt(0)
        | transaction_Level_Features["relinquished_calculated_player_count"].gt(0)
    ).sum(),
)

Acquired sides with calculated players: 1795
Relinquished sides with calculated players: 1795
Transactions with either side calculated: 2194


In [34]:
print("Index type:", type(transactions_In_Scope.index))

print("Index range:", transactions_In_Scope.index.min(), "to", transactions_In_Scope.index.max())

print("Rows:", len(transactions_In_Scope))

Index type: <class 'pandas.core.indexes.range.RangeIndex'>
Index range: 0 to 2748
Rows: 2749


In [35]:
# Rebuild stable transaction IDs before the final merge.
transactions_In_Scope = transactions_In_Scope.copy()

if "transaction_row_id" not in transactions_In_Scope.columns:
    transactions_In_Scope.insert(0, "transaction_row_id", transactions_In_Scope.index.to_numpy())

transactions_In_Scope = transactions_In_Scope.reset_index(drop=True)

In [36]:
print(transactions_In_Scope["transaction_row_id"].head())

print("Unique transaction IDs:", transactions_In_Scope["transaction_row_id"].nunique())

print("Audit IDs:", transaction_Feature_Audit["transaction_row_id"].nunique())

0    0
1    1
2    2
3    3
4    4
Name: transaction_row_id, dtype: int64
Unique transaction IDs: 2749
Audit IDs: 2749


In [37]:
transaction_ids = set(transactions_In_Scope["transaction_row_id"])

audit_ids = set(transaction_Feature_Audit["transaction_row_id"])

player_feature_ids = set(transaction_Player_Features["transaction_row_id"])

print("Transaction/audit overlap:", len(transaction_ids & audit_ids))

print("Transaction/player overlap:", len(transaction_ids & player_feature_ids))

print("Transaction IDs absent from audit:", len(transaction_ids - audit_ids))

print("Audit IDs absent from transactions:", len(audit_ids - transaction_ids))

Transaction/audit overlap: 2749
Transaction/player overlap: 2653
Transaction IDs absent from audit: 0
Audit IDs absent from transactions: 0


In [38]:
assert set(acquired_Transaction_Features["transaction_row_id"]).issubset(transaction_ids)

assert set(relinquished_Transaction_Features["transaction_row_id"]).issubset(transaction_ids)

print("Aggregated side identifiers aligned.")

Aggregated side identifiers aligned.


In [39]:
# Merge the validated side aggregates at transaction-row grain.
transaction_Level_Features = transactions_In_Scope.merge(
    acquired_Transaction_Features, on="transaction_row_id", how="left", validate="one_to_one"
).merge(relinquished_Transaction_Features, on="transaction_row_id", how="left", validate="one_to_one")

In [40]:
# Join point-in-time draft tiers at transaction-row grain.
# Join point-in-time draft tiers at the source transaction-team-row grain.
draft_Tier_Summary = pd.read_parquet("../data/interim/transaction_draft_tier_summary.parquet")

transaction_Level_Features = transaction_Level_Features.merge(
    draft_Tier_Summary, left_on="index", right_on="source_transaction_index", how="left", validate="one_to_one"
)

draft_tier_columns = [column for column in transaction_Level_Features.columns if column.startswith("draft_tier__")]
transaction_Level_Features[draft_tier_columns] = transaction_Level_Features[draft_tier_columns].fillna(0)

In [41]:
print("Source rows:", len(transactions_In_Scope))

print("Merged rows:", len(transaction_Level_Features))

print("Unique merged IDs:", transaction_Level_Features["transaction_row_id"].nunique())

Source rows: 2749
Merged rows: 2749
Unique merged IDs: 2749


In [42]:
side_count_columns = [
    column
    for column in transaction_Level_Features.columns
    if (column.startswith("acquired_") or column.startswith("relinquished_"))
    and (column.endswith("_count") or column.endswith("_players_with_data"))
]

transaction_Level_Features[side_count_columns] = transaction_Level_Features[side_count_columns].fillna(0).astype("int64")

In [43]:
source_calculated_players = transaction_Player_Features["feature_status"].eq("calculated").sum()

merged_calculated_players = (
    transaction_Level_Features["acquired_calculated_player_count"].sum()
    + transaction_Level_Features["relinquished_calculated_player_count"].sum()
)

print("Source calculated players:", source_calculated_players)

print("Merged calculated players:", merged_calculated_players)

assert source_calculated_players == merged_calculated_players

Source calculated players: 5194
Merged calculated players: 5194


In [44]:
transaction_Level_Features["acquired_has_calculated_player"] = transaction_Level_Features["acquired_calculated_player_count"] > 0

transaction_Level_Features["relinquished_has_calculated_player"] = transaction_Level_Features["relinquished_calculated_player_count"] > 0

In [45]:
transaction_Level_Features["both_sides_have_calculated_players"] = (
    transaction_Level_Features["acquired_has_calculated_player"] & transaction_Level_Features["relinquished_has_calculated_player"]
)

transaction_Level_Features["only_acquired_side_has_calculated_players"] = (
    transaction_Level_Features["acquired_has_calculated_player"] & ~transaction_Level_Features["relinquished_has_calculated_player"]
)

transaction_Level_Features["only_relinquished_side_has_calculated_players"] = (
    ~transaction_Level_Features["acquired_has_calculated_player"] & transaction_Level_Features["relinquished_has_calculated_player"]
)

transaction_Level_Features["neither_side_has_calculated_players"] = (
    ~transaction_Level_Features["acquired_has_calculated_player"] & ~transaction_Level_Features["relinquished_has_calculated_player"]
)

In [46]:
coverage_columns = [
    "both_sides_have_calculated_players",
    "only_acquired_side_has_calculated_players",
    "only_relinquished_side_has_calculated_players",
    "neither_side_has_calculated_players",
]

coverage_category_count = transaction_Level_Features[coverage_columns].astype(int).sum(axis=1)

print(coverage_category_count.value_counts())

print(transaction_Level_Features[coverage_columns].sum())

assert coverage_category_count.eq(1).all()

1    2749
Name: count, dtype: int64
both_sides_have_calculated_players               1396
only_acquired_side_has_calculated_players         399
only_relinquished_side_has_calculated_players     399
neither_side_has_calculated_players               555
dtype: int64


In [47]:
transaction_Level_Features["player_feature_eligible"] = (
    transaction_Level_Features["acquired_has_calculated_player"] | transaction_Level_Features["relinquished_has_calculated_player"]
)

In [48]:
transaction_Level_Features["asset_only_transaction"] = transaction_Level_Features["acquired_listed_player_count"].eq(
    0
) & transaction_Level_Features["relinquished_listed_player_count"].eq(0)

In [49]:
print("Player-feature eligible:", transaction_Level_Features["player_feature_eligible"].sum())

print("Neither side calculated:", transaction_Level_Features["neither_side_has_calculated_players"].sum())

print("Asset-only transactions:", transaction_Level_Features["asset_only_transaction"].sum())

Player-feature eligible: 2194
Neither side calculated: 555
Asset-only transactions: 96


In [50]:
# Pair acquired and relinquished features for net calculations.
net_feature_data = {}

acquired_feature_columns = [column for column in transaction_Level_Features.columns if column.startswith("acquired_")]

In [51]:
# Calculate acquired-minus-relinquished net features.
for acquired_column in acquired_feature_columns:
    feature_suffix = acquired_column[len("acquired_") :]

    relinquished_column = f"relinquished_{feature_suffix}"

    if relinquished_column not in (transaction_Level_Features.columns):
        continue

    acquired_values = transaction_Level_Features[acquired_column]

    relinquished_values = transaction_Level_Features[relinquished_column]

    if pd.api.types.is_bool_dtype(acquired_values) or pd.api.types.is_bool_dtype(relinquished_values):
        continue

    if not pd.api.types.is_numeric_dtype(acquired_values) or not pd.api.types.is_numeric_dtype(relinquished_values):
        continue

    net_feature_data[f"net_{feature_suffix}"] = acquired_values - relinquished_values

In [52]:
net_Transaction_Features = pd.DataFrame(net_feature_data, index=transaction_Level_Features.index)

transaction_Level_Features = pd.concat([transaction_Level_Features, net_Transaction_Features], axis=1)

In [53]:
print("Net features created:", len(net_Transaction_Features.columns))

print("Final transaction-level columns:", len(transaction_Level_Features.columns))

Net features created: 324
Final transaction-level columns: 1034


In [54]:
net_validation_columns = [
    "calculated_player_count",
    "season_minutes_sum",
    "season_points_per_100",
    "career_games_played_sum",
    "last10_game_score_average",
]

for feature in net_validation_columns:
    acquired_column = f"acquired_{feature}"
    relinquished_column = f"relinquished_{feature}"
    net_column = f"net_{feature}"

    if all(column in transaction_Level_Features.columns for column in [acquired_column, relinquished_column, net_column]):
        expected_values = transaction_Level_Features[acquired_column] - transaction_Level_Features[relinquished_column]

        assert np.allclose(transaction_Level_Features[net_column], expected_values, equal_nan=True)

print("Net features validated.")

Net features validated.


In [55]:
print("Rows:", len(transaction_Level_Features))

print("Unique transaction IDs:", transaction_Level_Features["transaction_row_id"].nunique())

print("Duplicate transaction IDs:", transaction_Level_Features["transaction_row_id"].duplicated().sum())

print("Duplicate column names:", transaction_Level_Features.columns.duplicated().sum())

Rows: 2749
Unique transaction IDs: 2749
Duplicate transaction IDs: 0
Duplicate column names: 0


In [56]:
# Calculate games played before and after each transaction.
transaction_Level_Features["pre_transaction_games"] = (
    transaction_Level_Features["current_Wins"] + transaction_Level_Features["current_Losses"]
)

transaction_Level_Features["post_transaction_games"] = (
    transaction_Level_Features["remaining_Wins"] + transaction_Level_Features["remaining_Losses"]
)

In [57]:
# Measure the change in team win percentage.
transaction_Level_Features["pre_transaction_win_percentage"] = transaction_Level_Features["current_Wins"] / transaction_Level_Features[
    "pre_transaction_games"
].replace(0, np.nan)

transaction_Level_Features["post_transaction_win_percentage"] = transaction_Level_Features["remaining_Wins"] / transaction_Level_Features[
    "post_transaction_games"
].replace(0, np.nan)

In [58]:
transaction_Level_Features["target_win_percentage_change"] = (
    transaction_Level_Features["post_transaction_win_percentage"] - transaction_Level_Features["pre_transaction_win_percentage"]
)

In [59]:
transaction_Level_Features["pre_transaction_82_game_win_pace"] = 82 * transaction_Level_Features["pre_transaction_win_percentage"]

transaction_Level_Features["post_transaction_82_game_win_pace"] = 82 * transaction_Level_Features["post_transaction_win_percentage"]

transaction_Level_Features["target_82_game_win_pace_change"] = 82 * transaction_Level_Features["target_win_percentage_change"]

In [60]:
transaction_Level_Features["expected_remaining_wins_at_pre_transaction_rate"] = (
    transaction_Level_Features["pre_transaction_win_percentage"] * transaction_Level_Features["post_transaction_games"]
)

transaction_Level_Features["target_remaining_wins_above_pre_transaction_rate"] = (
    transaction_Level_Features["remaining_Wins"] - transaction_Level_Features["expected_remaining_wins_at_pre_transaction_rate"]
)

In [61]:
# Flag rows with a usable model target.
transaction_Level_Features["target_available"] = (
    transaction_Level_Features["pre_transaction_games"].gt(0)
    & transaction_Level_Features["post_transaction_games"].gt(0)
    & transaction_Level_Features["target_win_percentage_change"].notna()
)

In [62]:
transaction_Level_Features["initial_model_eligible"] = (
    transaction_Level_Features["target_available"] & transaction_Level_Features["player_feature_eligible"]
)

In [63]:
transaction_Level_Features["at_least_5_pre_transaction_games"] = transaction_Level_Features["pre_transaction_games"] >= 5

transaction_Level_Features["at_least_10_pre_transaction_games"] = transaction_Level_Features["pre_transaction_games"] >= 10

In [64]:
# Apply the minimum-game stability requirements.
transaction_Level_Features["stable_target_eligible"] = (
    transaction_Level_Features["initial_model_eligible"] & transaction_Level_Features["at_least_10_pre_transaction_games"]
)

In [65]:
print("Target available:", transaction_Level_Features["target_available"].sum())

print("Initial model eligible:", transaction_Level_Features["initial_model_eligible"].sum())

print("Stable target eligible:", transaction_Level_Features["stable_target_eligible"].sum())

Target available: 937
Initial model eligible: 911
Stable target eligible: 857


In [66]:
transaction_Level_Features[["pre_transaction_games", "post_transaction_games"]].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

,pre_transaction_games,post_transaction_games
count,941.000000,941.000000
mean,41.366631,39.782147
std,16.038136,15.911535
min,0.000000,22.000000
1%,2.000000,23.000000
5%,7.000000,26.000000
10%,14.000000,27.000000
25%,32.000000,28.000000
50%,50.000000,31.000000
75%,53.000000,49.000000


In [67]:
transaction_Level_Features.loc[
    transaction_Level_Features["target_available"],
    ["target_win_percentage_change", "target_82_game_win_pace_change", "target_remaining_wins_above_pre_transaction_rate"],
].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

,target_win_percentage_change,target_82_game_win_pace_change,target_remaining_wins_above_pre_transaction_rate
count,937.000000,937.000000,937.000000
mean,0.010386,0.851691,0.543051
std,0.140308,11.505223,7.488246
min,-0.650000,-53.300000,-52.000000
1%,-0.336713,-27.610428,-22.872000
5%,-0.203668,-16.700806,-7.946667
10%,-0.156923,-12.867692,-5.591538
25%,-0.075789,-6.214737,-2.555556
50%,0.007500,0.615000,0.285714
75%,0.101648,8.335165,3.309091


In [68]:
invalid_record_rows = transaction_Level_Features.loc[
    transaction_Level_Features["current_Wins"].lt(0)
    | transaction_Level_Features["current_Losses"].lt(0)
    | transaction_Level_Features["remaining_Wins"].lt(0)
    | transaction_Level_Features["remaining_Losses"].lt(0)
    | transaction_Level_Features["pre_transaction_win_percentage"].lt(0)
    | transaction_Level_Features["pre_transaction_win_percentage"].gt(1)
    | transaction_Level_Features["post_transaction_win_percentage"].lt(0)
    | transaction_Level_Features["post_transaction_win_percentage"].gt(1)
]

print("Invalid team-record rows:", len(invalid_record_rows))

Invalid team-record rows:

 0


In [69]:
unavailable_Target_Rows = transaction_Level_Features.loc[
    ~transaction_Level_Features["target_available"],
    [
        "transaction_row_id",
        "Date",
        "Team",
        "Season",
        "current_Wins",
        "current_Losses",
        "pre_transaction_games",
        "remaining_Wins",
        "remaining_Losses",
        "post_transaction_games",
    ],
]

unavailable_Target_Rows

,transaction_row_id,Date,Team,Season,current_Wins,current_Losses,pre_transaction_games,remaining_Wins,remaining_Losses,post_transaction_games
14,14,1986-06-16,76ers,Offseason,NaN,NaN,NaN,NaN,NaN,NaN
15,15,1986-06-16,76ers,Offseason,NaN,NaN,NaN,NaN,NaN,NaN
16,16,1986-06-16,Bullets,Offseason,NaN,NaN,NaN,NaN,NaN,NaN
17,17,1986-06-16,Cavaliers,Offseason,NaN,NaN,NaN,NaN,NaN,NaN
18,18,1986-06-17,Trail Blazers,Offseason,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2683,2683,2018-08-31,Suns,Offseason,NaN,NaN,NaN,NaN,NaN,NaN
2684,2684,2018-10-15,Bucks,Offseason,NaN,NaN,NaN,NaN,NaN,NaN
2685,2685,2018-10-15,Wizards,Offseason,NaN,NaN,NaN,NaN,NaN,NaN
2686,2686,2018-10-15,Clippers,Offseason,NaN,NaN,NaN,NaN,NaN,NaN


In [70]:
# Record why a target is unavailable.
transaction_Level_Features["target_unavailable_reason"] = np.select(
    [
        transaction_Level_Features["pre_transaction_games"].eq(0),
        transaction_Level_Features["post_transaction_games"].eq(0),
        transaction_Level_Features[["current_Wins", "current_Losses", "remaining_Wins", "remaining_Losses"]].isna().any(axis=1),
    ],
    ["no_pre_transaction_games", "no_post_transaction_games", "missing_team_record"],
    default=pd.NA,
)

In [71]:
transaction_Level_Features.loc[
    ~transaction_Level_Features["target_available"],
    ["transaction_row_id", "Date", "Team", "Season", "pre_transaction_games", "post_transaction_games", "target_unavailable_reason"],
]

,transaction_row_id,Date,Team,Season,pre_transaction_games,post_transaction_games,target_unavailable_reason
14,14,1986-06-16,76ers,Offseason,NaN,NaN,missing_team_record
15,15,1986-06-16,76ers,Offseason,NaN,NaN,missing_team_record
16,16,1986-06-16,Bullets,Offseason,NaN,NaN,missing_team_record
17,17,1986-06-16,Cavaliers,Offseason,NaN,NaN,missing_team_record
18,18,1986-06-17,Trail Blazers,Offseason,NaN,NaN,missing_team_record
...,...,...,...,...,...,...,...
2683,2683,2018-08-31,Suns,Offseason,NaN,NaN,missing_team_record
2684,2684,2018-10-15,Bucks,Offseason,NaN,NaN,missing_team_record
2685,2685,2018-10-15,Wizards,Offseason,NaN,NaN,missing_team_record
2686,2686,2018-10-15,Clippers,Offseason,NaN,NaN,missing_team_record


In [72]:
# Create the processed-data output directory.
from pathlib import Path

processed_data_path = Path("../data/processed")

processed_data_path.mkdir(parents=True, exist_ok=True)

In [73]:
# Save the unified in-season and offseason transaction features.
transaction_Level_Features.to_parquet(processed_data_path / "transaction_level_features_offseason.parquet", index=False)

In [74]:
# Build a manifest describing every generated feature.
feature_Manifest = pd.DataFrame(
    {
        "column": transaction_Level_Features.columns,
        "dtype": [str(dtype) for dtype in transaction_Level_Features.dtypes],
        "missing_count": [transaction_Level_Features[column].isna().sum() for column in transaction_Level_Features.columns],
        "missing_percentage": [100 * transaction_Level_Features[column].isna().mean() for column in transaction_Level_Features.columns],
        "unique_count": [transaction_Level_Features[column].nunique(dropna=True) for column in transaction_Level_Features.columns],
    }
)

In [75]:
feature_Manifest["column_group"] = np.select(
    [
        feature_Manifest["column"].str.startswith("acquired_"),
        feature_Manifest["column"].str.startswith("relinquished_"),
        feature_Manifest["column"].str.startswith("net_"),
        feature_Manifest["column"].str.startswith("target_"),
        feature_Manifest["column"].isin(
            [
                "remaining_Wins",
                "remaining_Losses",
                "post_transaction_games",
                "post_transaction_win_percentage",
                "post_transaction_82_game_win_pace",
            ]
        ),
    ],
    ["acquired_player_features", "relinquished_player_features", "net_player_features", "target", "post_transaction_outcome"],
    default="transaction_context",
)

In [76]:
feature_Manifest.to_parquet(processed_data_path / "transaction_feature_manifest_offseason.parquet", index=False)

# Additive player-value preparation

The original transaction aggregation is complete above. The following cells validate and export the player-level point-in-time production inputs used by the additive model.


In [77]:
# Select player quality metrics used by draft-value modeling.
player_quality_metric_columns = [
    "season_points_per_100",
    "season_true_shooting_percentage",
    "season_assists_per_100",
    "season_turnover_percentage",
    "season_total_rebound_percentage",
    "season_steal_percentage",
    "season_block_percentage",
    "season_plus_minus_per_100",
    "season_game_score_average",
]

player_value_required_columns = [
    "transaction_row_id",
    "transaction_side",
    "transaction_player_name",
    "player_id",
    "feature_status",
    "player_match_status",
    "season_games_played",
    "season_minutes",
    "season_estimated_player_possessions",
    *player_quality_metric_columns,
]

missing_player_value_columns = [column for column in player_value_required_columns if column not in transaction_Player_Features.columns]

if missing_player_value_columns:
    raise KeyError("transaction_Player_Features is missing required additive-value inputs: " f"{missing_player_value_columns}")

print("Required player-value columns are present.")

Required player-value columns are present.


In [78]:
# Attach transaction context to player-level feature rows.
transaction_context_lookup = (
    transactions_In_Scope[["transaction_row_id", "Date", "Season", "Team"]]
    .rename(columns={"Date": "transaction_date", "Season": "transaction_season", "Team": "transaction_team"})
    .copy()
)

if not transaction_context_lookup["transaction_row_id"].is_unique:
    raise ValueError("transaction_row_id is not unique in transactions_In_Scope.")

player_value_input = transaction_Player_Features.copy()

for column in ["transaction_date", "transaction_season", "transaction_team"]:
    if column in player_value_input.columns:
        player_value_input = player_value_input.drop(columns=column)

player_value_input = player_value_input.merge(transaction_context_lookup, on="transaction_row_id", how="left", validate="many_to_one")

player_value_input["transaction_date"] = pd.to_datetime(player_value_input["transaction_date"], errors="coerce")

if player_value_input["transaction_date"].isna().any():
    raise ValueError("Some player rows did not receive a transaction date.")

In [79]:
numeric_player_value_columns = [
    "season_games_played",
    "season_minutes",
    "season_estimated_player_possessions",
    *player_quality_metric_columns,
]

for column in numeric_player_value_columns:
    player_value_input[column] = pd.to_numeric(player_value_input[column], errors="coerce")

calculated_rows = player_value_input["feature_status"].eq("calculated")
complete_calculated_rows = (
    calculated_rows
    & player_value_input[numeric_player_value_columns].notna().all(axis=1)
    & player_value_input["season_games_played"].gt(0)
    & player_value_input["season_minutes"].gt(0)
    & player_value_input["season_estimated_player_possessions"].gt(0)
)

known_zero_rows = player_value_input["player_match_status"].eq("no_prior_nba_appearance") | player_value_input["feature_status"].eq(
    "no_prior_box_score_appearance"
)

player_value_input["pre_model_player_value_status"] = np.select(
    [complete_calculated_rows, known_zero_rows],
    ["calculated_complete", "known_zero_no_prior_nba_production"],
    default="unresolved_or_incomplete",
)

player_value_input["pre_model_player_value_status"].value_counts(dropna=False)

pre_model_player_value_status
calculated_complete                   4978
known_zero_no_prior_nba_production     855
unresolved_or_incomplete               635
Name: count, dtype: int64

In [80]:
# Save model-ready player and transaction datasets.
processed_data_path = Path("../data/processed")
processed_data_path.mkdir(parents=True, exist_ok=True)

player_value_output_path = processed_data_path / "transaction_player_features_for_draft_value.parquet"
transaction_output_path = processed_data_path / "transaction_level_features_offseason.parquet"

player_value_input.to_parquet(player_value_output_path, index=False)
transaction_Level_Features.to_parquet(transaction_output_path, index=False)

print("Saved:", player_value_output_path)
print("Saved:", transaction_output_path)
print("Player rows:", len(player_value_input))
print("Transaction rows:", len(transaction_Level_Features))

Saved: ..\data\processed\transaction_player_features_for_draft_value.parquet
Saved: ..\data\processed\transaction_level_features_offseason.parquet
Player rows: 6468
Transaction rows: 2749
